# Subgrids: integer subsampling

A *subgrid* is a coarser grid whose points are a subset of a parent `Grid`'s points.
`SubGrid` builds one: along each `Dimension` subsampled by `factor`, it keeps every
`factor`-th point of the parent, so subgrid point `i` sits at parent index
`i*factor`.

Because the points coincide with parent points, no interpolation is involved. Devito
compiles a subgrid write to a **strided loop** -- `x += factor` -- rather than
visiting every point under an `if (x%factor == 0)` guard.

Every section below prints the generated C so you can see exactly what each case
produces.

## Creating a SubGrid

`SubGrid(parent, factor)` derives everything from the parent: the dimensions, the
shape, and the physical spacing.

In [1]:
import numpy as np
from devito import (CondEq, ConditionalDimension, Eq, Function, Grid, Operator,
                    SubGrid, TimeFunction)
from devito.ir.iet import FindNodes, Iteration

grid = Grid(shape=(12, 12), extent=(11., 11.))
x, y = grid.dimensions
sub = SubGrid(grid, factor=3)

sub

SubGrid[shape=(4, 4), factor=(3, 3)]

The shape is `ceil(12/3) = 4` along each axis, and the spacing is three times
the parent's -- the subgrid spans the same physical extent with fewer points.

In [2]:
print('subgrid shape  ', sub.shape, ' spacing', tuple(float(s) for s in sub.spacing))
print('parent  shape  ', grid.shape, ' spacing', tuple(float(s) for s in grid.spacing))

subgrid shape   (4, 4)  spacing (3.0, 3.0)
parent  shape   (12, 12)  spacing (1.0, 1.0)


Its dimensions are `ConditionalDimension`s derived from the parent's, which is
what carries the subsampling factor into the generated code.

In [3]:
sub.dimensions, sub.factor

((xs, ys), (3, 3))

A `Function` on a `SubGrid` needs neither `shape` nor `dimensions` -- unlike the
hand-rolled equivalent, where both must be worked out by hand.

In [4]:
u = Function(name='u', grid=grid)                 # parent grid
usub = Function(name='usub', grid=sub, space_order=0)   # subgrid

u.shape, usub.shape, usub.dimensions

((12, 12), (4, 4), (xs, ys))

A factor per `Dimension` subsamples each axis independently; `1` leaves an axis
untouched.

In [5]:
mixed = SubGrid(grid, factor=(3, 1))
mixed.shape, mixed.dimensions

((4, 12), (xs, y))

### Equivalent, by hand

`SubGrid` is a convenience over `ConditionalDimension`, which can still be used
directly when the dimensions need individual treatment. The two are equivalent; the
manual form just requires computing the shape yourself.

In [6]:
xs_manual = ConditionalDimension(name='xs_m', parent=grid.dimensions[0], factor=3)
f_manual = Function(name='f_manual', grid=grid,
                    dimensions=(xs_manual, grid.dimensions[1]),
                    shape=(4, 12), space_order=0)      # shape by hand

f_manual.shape == usub.shape

False

Throughout, this helper prints the loop nest of an `Operator` -- the part we care
about -- rather than the whole kernel.

In [7]:
def show(op):
    """Print the Operator's loop nest."""
    print(op.body.body[-1])


def steps(op):
    """The loop increment of each Iteration, by Dimension name."""
    return {i.dim.name: i.limits[2] for i in FindNodes(Iteration).visit(op)}

## The generated loop

`Eq(usub, u)` reads the parent grid and writes the subgrid.

In [8]:
# NBVAL_IGNORE_OUTPUT
op = Operator(Eq(usub, u))
show(op)

START(section0)
for (int x = x_m + (xsf - (x_m)%(xsf))%(xsf); x <= x_M; x += xsf)
{
  #pragma omp simd aligned(u,usub:64)
  for (int y = y_m + (ysf - (y_m)%(ysf))%(ysf); y <= y_M; y += ysf)
  {
    usub[x / xsf][y / ysf] = u[x + 1][y + 1];
  }
}
STOP(section0,timers)


Two features of the `x` loop header matter:

- the increment is `x += xsf`, so only one point in three is visited -- there is no
  `if` anywhere in the nest;
- the minimum is `x_m + (xsf - x_m%xsf)%xsf` rather than `x_m`.

That second expression is the **anchor**. A subgrid point is one where
`x % factor == 0`, and `x_m` is not necessarily such a point, so iteration must start
at the first subgrid point at or after `x_m`.

The body carries the index conversion `usub[x / xsf]`, mapping the parent coordinate
to a subgrid coordinate.

In [9]:
steps(op)

{'x': xsf, 'y': ysf}

### Checking the result

`usub` should hold every third row of `u`.

In [10]:
# NBVAL_IGNORE_OUTPUT
u.data[:] = np.arange(144).reshape(12, 12)
usub.data[:] = 0.
Operator(Eq(usub, u)).apply()

assert np.array_equal(usub.data, u.data[::3, ::3])
usub.data

Operator `Kernel` ran in 0.01 s


Data([[  0.,   3.,   6.,   9.],
      [ 36.,  39.,  42.,  45.],
      [ 72.,  75.,  78.,  81.],
      [108., 111., 114., 117.]], dtype=float32)

### The anchor in action

Run with `x_m=2`, so iteration no longer starts on a multiple of 3. The first subgrid
point is `x=3`, so the slot for `x=0` is never written.

In [11]:
# NBVAL_IGNORE_OUTPUT
u.data[:] = 1.
usub.data[:] = -1.                      # sentinel: never written
Operator(Eq(usub, u)).apply(x_m=2)

usub.data[:, 0]

Operator `Kernel` ran in 0.01 s


Data([-1.,  1.,  1.,  1.], dtype=float32)

The leading `-1.` is the untouched slot. Had iteration started at `x_m=2` instead
of `x=3`, the wrong points would have been written.

## Subsampling the innermost Dimension

Subsampling `y`, the innermost (vectorized) `Dimension`, is strided too. The
alternative -- a guard inside the SIMD loop -- would mean a predicated store on every
iteration, which is worse.

In [12]:
# NBVAL_IGNORE_OUTPUT
sub_y = SubGrid(grid, factor=(1, 2))      # coarsen y only
usub_y = Function(name='usub_y', grid=sub_y, space_order=0)

op_inner = Operator(Eq(usub_y, u))
show(op_inner)

START(section0)
for (int x = x_m; x <= x_M; x += 1)
{
  #pragma omp simd aligned(u,usub_y:64)
  for (int y = y_m + (ysf - (y_m)%(ysf))%(ysf); y <= y_M; y += ysf)
  {
    usub_y[x][y / ysf] = u[x + 1][y + 1];
  }
}
STOP(section0,timers)


In [13]:
steps(op_inner)

{'x': 1, 'y': ysf}

The `y` loop is strided and the `#pragma omp simd` is retained.

In [14]:
# NBVAL_IGNORE_OUTPUT
u.data[:] = np.arange(144).reshape(12, 12)
Operator(Eq(usub_y, u)).apply()

assert np.array_equal(usub_y.data, u.data[:, ::2])
usub_y.data[:3]

Operator `Kernel` ran in 0.01 s


Data([[ 0.,  2.,  4.,  6.,  8., 10.],
      [12., 14., 16., 18., 20., 22.],
      [24., 26., 28., 30., 32., 34.]], dtype=float32)

## Subsampling several Dimensions

Each subsampled `Dimension` gets its own stride, independently.

In [15]:
# NBVAL_IGNORE_OUTPUT
sub_xy = SubGrid(grid, factor=(3, 2))
usub_xy = Function(name='usub_xy', grid=sub_xy, space_order=0)

op_both = Operator(Eq(usub_xy, u))
show(op_both)

START(section0)
for (int x = x_m + (xsf - (x_m)%(xsf))%(xsf); x <= x_M; x += xsf)
{
  #pragma omp simd aligned(u,usub_xy:64)
  for (int y = y_m + (ysf - (y_m)%(ysf))%(ysf); y <= y_M; y += ysf)
  {
    usub_xy[x / xsf][y / ysf] = u[x + 1][y + 1];
  }
}
STOP(section0,timers)


In [16]:
# NBVAL_IGNORE_OUTPUT
u.data[:] = np.arange(144).reshape(12, 12)
Operator(Eq(usub_xy, u)).apply()

assert np.array_equal(usub_xy.data, u.data[::3, ::2])
steps(op_both)

Operator `Kernel` ran in 0.01 s


{'x': xsf, 'y': ysf}

## When a guard is used instead

Striding is applied only where it provably preserves semantics. Where it does not,
Devito visits every point under an `if (x%factor == 0)` guard. Results are the same
either way; only the generated loop differs. Four cases, each with its code.

### 1. The loop nest is shared with an unguarded equation

`Eq(f, 1)` must run at **every** `x`, while `Eq(g, f)` runs every fourth. Both land in
one loop nest, so striding it would also stride `Eq(f, 1)`. The guard is kept.

In [17]:
# NBVAL_IGNORE_OUTPUT
grid1 = Grid(shape=(12,))
(x1,) = grid1.dimensions
xs1 = ConditionalDimension(name='xs1', parent=x1, factor=4)

f = Function(name='f', grid=grid1)
g = Function(name='g', grid=grid1, dimensions=(xs1,), shape=(3,), space_order=0)

op_shared = Operator([Eq(f, 1), Eq(g, f)])
show(op_shared)

START(section0)
for (int x = x_m; x <= x_M; x += 1)
{
  f[x + 1] = 1;
  if ((x)%(xs1f) == 0)
  {
    g[x / xs1f] = f[x + 1];
  }
}
STOP(section0,timers)


`f[x + 1] = 1` sits outside the `if`, `g[x / xs1f]` inside it -- which is exactly
what is required.

### 2. The subsampled Dimension is sequential

Striding skips points, which is only sound when no dependence is carried along the
`Dimension`. Here `g` reads its own previous subgrid value, so `x` is sequential.

In [18]:
# NBVAL_IGNORE_OUTPUT
grid2 = Grid(shape=(12, 12))
x2, y2 = grid2.dimensions
xs3 = ConditionalDimension(name='xs3', parent=x2, factor=3)

f2 = Function(name='f2', grid=grid2)
g2 = Function(name='g2', grid=grid2, dimensions=(xs3, y2), shape=(4, 12),
              space_order=0)

op_seq = Operator(Eq(g2, g2.subs(xs3, xs3-1) + f2))
show(op_seq)

START(section0)
for (int x = x_m; x <= x_M; x += 1)
{
  if ((x)%(xs3f) == 0)
  {
    for (int y = y_m; y <= y_M; y += 1)
    {
      g2[x / xs3f][y] = f2[x + 1][y + 1] + g2[(x / xs3f) - 1][y];
    }
  }
}
STOP(section0,timers)


Time subsampling is the everyday instance of this: the time loop is sequential, so
a saved-wavefield `Eq` keeps its guard.

In [19]:
# NBVAL_IGNORE_OUTPUT
nt, factor = 9, 4
grid3 = Grid(shape=(4, 4))
ts = ConditionalDimension(name='ts', parent=grid3.time_dim, factor=factor)

v = TimeFunction(name='v', grid=grid3)
vsave = TimeFunction(name='vsave', grid=grid3, time_dim=ts,
                     save=(nt + factor - 1)//factor)

op_time = Operator([Eq(v.forward, v + 1.), Eq(vsave, v)])
show(op_time)

for (int time = time_m, t0 = (time)%(2), t1 = (time + 1)%(2); time <= time_M; time += 1, t0 = (time)%(2), t1 = (time + 1)%(2))
{
  START(section0)
  for (int x = x_m; x <= x_M; x += 1)
  {
    #pragma omp simd aligned(v:64)
    for (int y = y_m; y <= y_M; y += 1)
    {
      v[t1][x + 1][y + 1] = v[t0][x + 1][y + 1] + 1.0F;
    }
  }
  STOP(section0,timers)

  if ((time)%(tsf) == 0)
  {
    START(section1)
    for (int x = x_m; x <= x_M; x += 1)
    {
      #pragma omp simd aligned(v,vsave:64)
      for (int y = y_m; y <= y_M; y += 1)
      {
        vsave[time / tsf][x + 1][y + 1] = v[t0][x + 1][y + 1];
      }
    }
    STOP(section1,timers)
  }
}


The saved slices are correct -- time subsampling behaves exactly as it always has,
it simply does not take the strided form.

In [20]:
# NBVAL_IGNORE_OUTPUT
op_time.apply(time_M=nt-2)

nwritten = (nt - 2)//factor + 1
for i in range(nwritten):
    assert np.allclose(vsave.data[i], i*factor)

[float(vsave.data[i, 0, 0]) for i in range(vsave.shape[0])]

Operator `Kernel` ran in 0.01 s


[0.0, 4.0, 0.0]

### 3. A `condition` is attached as well

A `ConditionalDimension` may carry an arbitrary `condition` alongside a `factor`. The
combined test is no longer "every `factor`-th point", so it is evaluated pointwise.

In [21]:
# NBVAL_IGNORE_OUTPUT
xs4 = ConditionalDimension(name='xs4', parent=x, factor=3,
                           condition=CondEq(x % 2, 0))
w = Function(name='w', grid=grid, dimensions=(xs4, y), shape=(4, 12),
             space_order=0)

op_cond = Operator(Eq(w, u))
show(op_cond)

START(section0)
for (int x = x_m; x <= x_M; x += 1)
{
  if ((x & 1) == 0 && (x)%(xs4f) == 0)
  {
    #pragma omp simd aligned(u,w:64)
    for (int y = y_m; y <= y_M; y += 1)
    {
      w[x / xs4f][y] = u[x + 1][y + 1];
    }
  }
}
STOP(section0,timers)


### 4. Accesses are not affine in the subsampled Dimension

An indirectly indexed read (`u[idx[x, y], y]`) is not affine in `x`, so the points a
stride would skip cannot be shown to be unneeded.

In [22]:
# NBVAL_IGNORE_OUTPUT
idx = Function(name='idx', grid=grid, dtype=np.int32)
gi = Function(name='gi', grid=sub, space_order=0)

op_indirect = Operator(Eq(gi, u[idx[x, y], y]))
show(op_indirect)

START(section0)
for (int x = x_m; x <= x_M; x += 1)
{
  if ((x)%(xsf) == 0)
  {
    #pragma omp simd aligned(gi,idx,u:64)
    for (int y = y_m; y <= y_M; y += 1)
    {
      if ((y)%(ysf) == 0)
      {
        int r0 = idx[x + 1][y + 1];
        gi[x / xsf][y / ysf] = u[r0 + 1][y + 1];
      }
    }
  }
}
STOP(section0,timers)


## Summary

Subsampling along a `Dimension` is declared with `ConditionalDimension(factor=n)` plus
a `Function` defined over it. Devito then generates, for that `Dimension`:

| | strided loop | guarded loop |
|---|---|---|
| loop header | `x += factor`, anchored | `x += 1` |
| body | unconditional | wrapped in `if (x%factor == 0)` |
| iterations | one in `factor` | all |
| used when | the Dimension is parallel and accesses are affine | otherwise |

The strided form is the default, including for the innermost (vectorized)
`Dimension` and for several subsampled `Dimension`s at once. The guarded form is the
fallback for the four cases above: a loop nest shared with an unguarded equation, a
sequential `Dimension` such as time, an extra `condition`, or non-affine accesses.
Both compute the same thing.

Generic subsampling -- subgrid points that fall *between* parent grid points and so
require interpolation -- is a separate topic, not covered here.